In [8]:
import gymnasium as gym
import numpy as np
import csv


class Agent:
    def __init__(self, n_states, n_actions, alpha, epsilon, gamma):
        self.alpha = alpha
        self.epsilon = epsilon
        self.gamma = gamma

        # Q-table (estado x ação)
        self.q = np.zeros((n_states, n_actions))

        # Guarda o estado atual (linha 6 da folha)
        self.prev_state = None

    def reset(self):
        self.prev_state = None

    def select_action(self, state):
        """
        Linha 6 da folha:
        action = agent.select_action(state)

        Implementa epsilon-greedy.
        Também salva o estado atual para o update.
        """
        self.prev_state = state

        if np.random.random() < self.epsilon:
            return np.random.randint(self.q.shape[1])

        return int(np.argmax(self.q[state]))

    def update(self, action, state, reward):
        """
        Linha 8 da folha:
        agent.update(action, state, reward)

        Aqui:
        - 'state' recebido é o next_state da linha 7.
        - O estado anterior está em self.prev_state.
        """

        s = self.prev_state     # estado da linha 6
        s_next = state          # estado da linha 7 (next_state)

        q_sa = self.q[s, action]

        # Se terminal, não existe próximo estado
        if s_next == -1:
            target = reward
        else:
            target = reward + self.gamma * np.max(self.q[s_next])

        # Equação de Bellman (Q-Learning)
        self.q[s, action] = q_sa + self.alpha * (target - q_sa)

    def save_q_table(self, filename):
        np.savetxt(filename, self.q, delimiter=",")


# -------------------------
# TREINAMENTO
# -------------------------

env = gym.make("Taxi-v3")

n_states = env.observation_space.n
n_actions = env.action_space.n

# teste 1
#alpha = 0.1
#epsilon = 0.1
#gamma = 0.99

#teste 2
alpha = 0.5
epsilon = 0.3
gamma = 0.95

agent = Agent(n_states, n_actions, alpha, epsilon, gamma)

MAX_EPISODE = 5000
T = 200

with open("log.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["episode", "reward"])

    for episode in range(MAX_EPISODE):

        # Linha 3
        state, _ = env.reset()

        # Linha 4
        agent.reset()

        episode_reward = 0

        # Linha 5
        for t in range(T):

            # Linha 6
            action = agent.select_action(state)

            # Linha 7
            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated

            episode_reward += reward

            # Linha 8
            if done:
                agent.update(action, -1, reward)  # terminal
            else:
                agent.update(action, next_state, reward)

            # Atualiza estado atual
            state = next_state

            if done:
                break

        writer.writerow([episode, episode_reward])

# Salva Q-table
agent.save_q_table("q_table.csv")

env.close()

print("Treinamento finalizado.")
print("Arquivos gerados: log.csv e q_table.csv")


# -------------------------
# EXECUÇÃO SEM TREINAMENTO
# -------------------------

print("\nExecutando agente treinado...\n")

# Carrega Q-table salva
q_loaded = np.loadtxt("q_table.csv", delimiter=",")

env = gym.make("Taxi-v3")

for ep in range(3):
    state, _ = env.reset()
    done = False
    total_reward = 0

    while not done:
        # SEM epsilon, só greedy
        action = int(np.argmax(q_loaded[state]))

        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated

        total_reward += reward
        state = next_state

    print(f"Episódio {ep} -> recompensa acumulada: {total_reward}")

env.close()

Treinamento finalizado.
Arquivos gerados: log.csv e q_table.csv

Executando agente treinado...

Episódio 0 -> recompensa acumulada: 9
Episódio 1 -> recompensa acumulada: 3
Episódio 2 -> recompensa acumulada: 11


In [9]:
import matplotlib.pyplot as plt

episodes = []
rewards = []

with open("log.csv", "r", encoding="utf-8") as f:
    next(f)  # pula o cabeçalho
    for line in f:
        ep, r = line.strip().split(",")
        episodes.append(int(ep))
        rewards.append(float(r))

plt.figure()
plt.plot(episodes, rewards)
plt.xlabel("Episódio")
plt.ylabel("Recompensa acumulada")
plt.title("Curva de aprendizado - Taxi Driver")
plt.savefig("learning_curve.png", dpi=150, bbox_inches="tight")
plt.close()

print("Gráfico salvo em learning_curve.png")

Gráfico salvo em learning_curve.png
